# 02 — Data Preprocessing & Quarantine Pipeline

**Project:** AI-Based IoT Sensor Anomaly Detection (ESP32 + DHT11)  
**Module:** [`preprocess/clean_datasets.py`](../../preprocess/clean_datasets.py)  

This notebook walks through reproducible preprocessing while keeping `data/raw/*.csv` **strictly immutable**:
1. Robust CSV loading (handles missing headers / blank line 1 in `normal_data.csv`).
2. Exact deduplication + same-second serial burst collapse.
3. Anomaly quarantine (`quarantine_unlabeled_anomalies`) separating unlabelled breath/step spikes.
4. Chronological Nov 1 clean holdout split + live `COM19` baseline integration into `eval_test_suite.csv`.
5. Pilot verified annotation rebuild (`23:09:38 -> 23:40:28` and `00:34:20 -> 00:35:35`).

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config.yaml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import config
print(f'Project root: {PROJECT_ROOT}')
import json
import pandas as pd
from preprocess.clean_datasets import load_raw_normal, quarantine_unlabeled_anomalies

summary_path = config.processed_dir / 'cleaning_summary.json'
cleaning_summary = json.loads(summary_path.read_text(encoding='utf-8'))
for k, v in cleaning_summary.items():
    print(f'{k:30s}: {v}')

Project root: c:\Users\Avinash Reddy\OneDrive\Desktop\Iot anomaly detection
raw_normal_rows               : 49728
dedup_exact_rows              : 36952
after_burst_collapse_rows     : 36663
quarantined_anomaly_rows      : 389
train_clean_normal_rows       : 34383
holdout_nov1_rows             : 1891
live_today_rows               : 59
pilot_rows                    : 3087
pilot_annotated_rows          : 504
eval_suite_rows               : 1950
eval_suite_anomalies          : 143
eval_suite_faults             : {'slow_drift_fault': 45, 'heat_source_burst': 24, 'cold_draft_drop': 20, 'offset_fault': 20, 'breath_spike': 18, 'chatter_noise_fault': 16}


## Step-by-Step Verification of `quarantine_unlabeled_anomalies()`
We verify on the deduplicated normal trace that quarantining removes sharp step jumps and extreme humidity spikes while leaving steady indoor diurnal curves completely intact.

In [2]:
raw_normal = load_raw_normal(config.raw_dir / 'normal_data.csv')
dedup = raw_normal.drop_duplicates().sort_values('timestamp', kind='stable').reset_index(drop=True)
no_burst = dedup.drop_duplicates(subset=['timestamp'], keep='last').reset_index(drop=True)
clean_all, quarantined = quarantine_unlabeled_anomalies(no_burst, gap_threshold=config.gap_threshold_sec)

print(f'After dedupe+burst : {len(no_burst):,}')
print(f'Clean normal all   : {len(clean_all):,} | H range=[{clean_all.humidity.min():.1f}, {clean_all.humidity.max():.1f}]%')
print(f'Quarantined rows   : {len(quarantined):,} | H range=[{quarantined.humidity.min():.1f}, {quarantined.humidity.max():.1f}]%')

After dedupe+burst : 36,663
Clean normal all   : 36,274 | H range=[46.8, 66.4]%
Quarantined rows   : 389 | H range=[51.6, 97.2]%


## Unseen Evaluation Suite & Live Hardware Telemetry
`data/processed/eval_test_suite.csv` combines 1,807 clean held-out Nov 1 baseline/fault rows plus 59 live `COM19` DHT11 rows collected on 2026-11-10 across 6 controlled anomaly regimes (`breath_spike`, `heat_source_burst`, `cold_draft_drop`, `offset_fault`, `slow_drift_fault`, `chatter_noise_fault`).

In [3]:
suite = pd.read_csv(config.processed_dir / 'eval_test_suite.csv', parse_dates=['timestamp'])
print(f'Eval suite total rows: {len(suite):,}')
print('\nBreakdown by fault_type:')
print(suite.groupby(['fault_type', 'label']).agg(
    rows=('temperature', 'count'),
    temp_min=('temperature', 'min'),
    temp_max=('temperature', 'max'),
    hum_min=('humidity', 'min'),
    hum_max=('humidity', 'max'),
).to_string())

Eval suite total rows: 1,950

Breakdown by fault_type:
                           rows  temp_min  temp_max  hum_min  hum_max
fault_type          label                                            
breath_spike        1        18      26.5      31.7     58.3     83.3
chatter_noise_fault 1        16      24.0      28.9     50.0     66.7
cold_draft_drop     1        20      21.7      26.5     58.0     67.0
heat_source_burst   1        24      26.5      34.0     43.1     58.1
live_dht11_normal   0        59      28.1      29.7     53.7     59.8
normal_holdout      0      1748      25.9      26.6     57.9     60.9
offset_fault        1        20      31.2      31.2     47.6     47.7
slow_drift_fault    1        45      27.0      31.9     60.3     76.2
